# Consumer360 — Week 1 Data Foundation Audit

A read-only audit of the Week 1 deliverables: source data reconciliation, PostgreSQL staging, data quality signals, transaction classification, star schema, referential integrity, Single Customer View, repository SQL assets, and historical SQL performance evidence.

The notebook does not modify source files or database objects. It uses the project's existing local configuration where available. Missing prerequisites are reported in the audit results; they do not require manual notebook edits.


## 1. Scope and reported baseline

Reference values from `Consumer360_Week1_Completion_and_Details.docx`:

| Measure | Reported value |
|---|---:|
| 2009–10 source rows | 525,461 |
| 2010–11 source rows | 541,910 |
| Cross-file matching occurrences | 22,523 |
| Staging / fact rows | 1,044,848 |
| `dim_customer` rows including unknown | 5,943 |
| `dim_product` rows including unknown | 5,305 |
| Identified customers in Single Customer View | 5,942 |
| Merchandise purchasers | 5,852 |
| Customers without merchandise orders | 90 |
| SCV merchandise revenue | £17,117,089.2970 |
| SQL performance target | under 2,000 ms |

The report says overlap was excluded from the second source file only; within-file duplicates were preserved. Source files were not modified.

In [ ]:
from pathlib import Path
import hashlib, sys
import pandas as pd
from IPython.display import display

# Find the project root from the notebook's current working directory or its parents.
_candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in _candidates if (p / "pyproject.toml").exists() and (p / "database").exists()), Path.cwd())
RAW = ROOT / "data" / "raw"
REPORT = ROOT / "reports" / "ingestion_report.json"
MIGRATIONS = ROOT / "database" / "migrations"
ERD = ROOT / "database" / "diagrams" / "consumer360_star_schema.mmd"
BENCHMARK = ROOT / "database" / "queries" / "006_performance_benchmark.sql"
FILES = [RAW / "online_retail_II_2009-10.csv", RAW / "online_retail_II_2010-11.csv"]
DATE_FORMAT = "%d-%m-%y %H:%M"
COLUMNS = ["Invoice", "StockCode", "Description", "Quantity", "InvoiceDate", "Price", "Customer ID", "Country"]

BASELINE = {"source_2009":525461, "source_2010":541910, "overlap":22523,
            "staging":1044848, "customers":5943, "products":5305, "fact":1044848,
            "scv_customers":5942, "purchasers":5852, "no_orders":90,
            "scv_revenue":17117089.2970, "target_ms":2000.0}
checks = []

def add_check(name, status, observed=None, expected=None, note=None):
    checks.append({"check":name,"status":status,"observed":observed,"expected":expected,"note":note})

def compare(name, observed, expected, tol=0):
    if observed is None or pd.isna(observed):
        status = "NOT VERIFIED"
    elif isinstance(expected, (int,float)):
        status = "PASS" if abs(float(observed)-float(expected)) <= tol else "REVIEW"
    else:
        status = "PASS" if observed == expected else "REVIEW"
    add_check(name,status,observed,expected)

print("Project root:", ROOT)
print("Python:", sys.version.split()[0])
from sqlalchemy import text


## 2. Source data audit

The eight expected fields are `Invoice`, `StockCode`, `Description`, `Quantity`, `InvoiceDate`, `Price`, `Customer ID`, and `Country`. Identifiers are loaded as text so numeric coercion cannot alter them.

In [ ]:
frames, source_rows = {}, []
for path in FILES:
    if not path.exists():
        add_check(f"Source file: {path.name}", "NOT VERIFIED", "Not found", "Available in data/raw")
        continue
    try:
        df = pd.read_csv(path, encoding="utf-8-sig", dtype="string", low_memory=False)
        frames[path.name] = df
        missing = sorted(set(COLUMNS) - set(df.columns))
        if not missing:
            dates = pd.to_datetime(df["InvoiceDate"], format=DATE_FORMAT, errors="coerce")
            min_date, max_date = dates.min(), dates.max()
        else:
            min_date = max_date = None
        source_rows.append({"file":path.name,"rows":len(df),"columns":len(df.columns),
            "missing_required_columns":missing,"min_date":min_date,"max_date":max_date,
            "size_bytes":path.stat().st_size,
            "sha256":hashlib.sha256(path.read_bytes()).hexdigest()})
        add_check(f"Required columns: {path.name}", "PASS" if not missing else "FAIL", missing, [])
    except Exception as exc:
        add_check(f"Read source file: {path.name}", "NOT VERIFIED", type(exc).__name__, "Readable CSV")

source_audit = pd.DataFrame(source_rows)
display(source_audit if not source_audit.empty else pd.DataFrame(columns=["file","rows","columns","missing_required_columns","min_date","max_date","size_bytes","sha256"]))
if len(source_rows) == 2:
    rows_by_file = {r["file"]: r["rows"] for r in source_rows}
    compare("2009–10 source row count", rows_by_file.get(FILES[0].name), BASELINE["source_2009"])
    compare("2010–11 source row count", rows_by_file.get(FILES[1].name), BASELINE["source_2010"])

if len(frames) == 2 and all(set(COLUMNS).issubset(frames[p.name].columns) for p in FILES):
    left, right = [frames[p.name][COLUMNS].fillna("<NULL>").astype(str) for p in FILES]
    lc, rc = left.value_counts(), right.value_counts()
    overlap = int(pd.concat([lc,rc],axis=1).fillna(0).min(axis=1).sum())
    retained = len(left) + len(right) - overlap
    print("Cross-file matching occurrences:", overlap)
    compare("Cross-file overlap occurrences", overlap, BASELINE["overlap"])
    compare("Retained rows after overlap handling", retained, BASELINE["staging"])
else:
    add_check("Cross-file overlap", "NOT VERIFIED", "Both readable files with required columns are needed", BASELINE["overlap"])


## 3. PostgreSQL connection and staging audit

The project's existing connection helper loads credentials from the local `.env` file. Credentials are never displayed. Database queries are read-only. If PostgreSQL is unavailable, database checks are marked `NOT VERIFIED` and the notebook continues.


In [ ]:
engine = None
try:
    sys.path.insert(0, str(ROOT / "src"))
    from consumer360.database.connection import get_engine
    engine = get_engine()
    with engine.connect() as conn:
        identity = pd.read_sql_query(text("SELECT current_database() AS database_name, current_user AS database_user, version() AS server_version"), conn)
    display(identity)
    add_check("PostgreSQL connection","PASS","Connected","Connected")
except Exception as exc:
    print("Connection unavailable:",type(exc).__name__,str(exc)[:250])
    add_check("PostgreSQL connection","NOT VERIFIED",type(exc).__name__,"Connected")

if engine is not None:
    try:
        with engine.connect() as conn:
            staging = pd.read_sql_query(text("""
                SELECT COUNT(*) AS row_count,
                       COUNT(DISTINCT staging_id) AS distinct_staging_ids,
                       COUNT(*) FILTER (WHERE customer_id IS NULL) AS missing_customer_id,
                       COUNT(*) FILTER (WHERE description IS NULL) AS missing_description,
                       COUNT(*) FILTER (WHERE unit_price = 0) AS zero_unit_price,
                       COUNT(*) FILTER (WHERE unit_price < 0) AS negative_unit_price,
                       COUNT(*) FILTER (WHERE quantity < 0) AS negative_quantity,
                       COUNT(*) FILTER (WHERE is_cancellation) AS cancellation_rows,
                       COUNT(*) FILTER (WHERE source_file IS NULL OR source_row_number IS NULL) AS missing_lineage,
                       MIN(invoice_date) AS first_invoice_date,
                       MAX(invoice_date) AS last_invoice_date
                FROM staging.retail_transactions
            """), conn)
        display(staging)
        s = staging.iloc[0]
        compare("Staging row count",int(s.row_count),BASELINE["staging"])
        compare("Staging ID uniqueness",int(s.distinct_staging_ids),int(s.row_count))
        compare("Staging lineage completeness",int(s.missing_lineage),0)
    except Exception as exc:
        print("Staging query unavailable:",type(exc).__name__,str(exc)[:250])
        add_check("Staging audit","NOT VERIFIED",type(exc).__name__,"staging.retail_transactions query")

## 4. Data quality and classification audit

Cancellation invoice flags and negative quantities are separate signals. The Week 1 report records 19,165 cancellation-flagged rows, 22,557 negative-quantity rows, 19,164 with both, 3,393 negative quantities without cancellation flags, and one cancellation flag without negative quantity.

The v2 classification is additive and retains all 1,044,848 source lines.

In [ ]:
if engine is not None:
    try:
        with engine.connect() as conn:
            signals = pd.read_sql_query(text("""
                SELECT COUNT(*) FILTER (WHERE is_cancellation) AS cancellation_flagged,
                       COUNT(*) FILTER (WHERE quantity < 0) AS negative_quantity,
                       COUNT(*) FILTER (WHERE is_cancellation AND quantity < 0) AS both_signals,
                       COUNT(*) FILTER (WHERE NOT is_cancellation AND quantity < 0) AS negative_qty_without_cancellation,
                       COUNT(*) FILTER (WHERE is_cancellation AND quantity >= 0) AS cancellation_without_negative_qty
                FROM staging.retail_transactions
            """), conn)
            categories = pd.read_sql_query(text("""
                SELECT source_category, reporting_category, COUNT(*) AS row_count
                FROM analytics.fact_sales
                GROUP BY source_category, reporting_category
                ORDER BY source_category, reporting_category
            """), conn)
        display(signals); display(categories)
        compare("Classification row preservation",int(categories.row_count.sum()),BASELINE["fact"])
    except Exception as exc:
        print("Classification audit unavailable:",type(exc).__name__,str(exc)[:250])
        add_check("Classification audit","NOT VERIFIED",type(exc).__name__,"Staging signals and fact categories")

## 5. Star schema and referential integrity audit

Expected objects: `analytics.dim_customer`, `analytics.dim_product`, and `analytics.fact_sales`. The fact grain is one row per staged source transaction line, with unique `staging_id`. The report records one unknown member (`key = 0`) in each dimension and foreign keys from fact to customer, product, and staging.

In [ ]:
if engine is not None:
    try:
        with engine.connect() as conn:
            counts = pd.read_sql_query(text("""
                SELECT 'analytics.dim_customer' AS object_name, COUNT(*) AS row_count FROM analytics.dim_customer
                UNION ALL SELECT 'analytics.dim_product', COUNT(*) FROM analytics.dim_product
                UNION ALL SELECT 'analytics.fact_sales', COUNT(*) FROM analytics.fact_sales
            """), conn)
            fact = pd.read_sql_query(text("""
                SELECT COUNT(*) AS fact_rows, COUNT(DISTINCT staging_id) AS distinct_staging_ids
                FROM analytics.fact_sales
            """), conn)
            unknown = pd.read_sql_query(text("""
                SELECT (SELECT COUNT(*) FROM analytics.dim_customer WHERE customer_key=0) AS unknown_customers,
                       (SELECT COUNT(*) FROM analytics.dim_product WHERE product_key=0) AS unknown_products
            """), conn)
            orphans = pd.read_sql_query(text("""
                SELECT COUNT(*) FILTER (WHERE c.customer_key IS NULL) AS orphan_customer_rows,
                       COUNT(*) FILTER (WHERE p.product_key IS NULL) AS orphan_product_rows,
                       COUNT(*) FILTER (WHERE s.staging_id IS NULL) AS orphan_staging_rows
                FROM analytics.fact_sales f
                LEFT JOIN analytics.dim_customer c ON f.customer_key=c.customer_key
                LEFT JOIN analytics.dim_product p ON f.product_key=p.product_key
                LEFT JOIN staging.retail_transactions s ON f.staging_id=s.staging_id
            """), conn)
        display(counts); display(fact); display(unknown); display(orphans)
        count_map = dict(zip(counts.object_name,counts.row_count))
        compare("dim_customer row count",int(count_map["analytics.dim_customer"]),BASELINE["customers"])
        compare("dim_product row count",int(count_map["analytics.dim_product"]),BASELINE["products"])
        compare("fact_sales row count",int(count_map["analytics.fact_sales"]),BASELINE["fact"])
        compare("Fact staging_id uniqueness",int(fact.distinct_staging_ids.iloc[0]),int(fact.fact_rows.iloc[0]))
        compare("Unknown customer member count",int(unknown.unknown_customers.iloc[0]),1)
        compare("Unknown product member count",int(unknown.unknown_products.iloc[0]),1)
        for col in orphans.columns:
            compare(f"Zero {col}",int(orphans[col].iloc[0]),0)
    except Exception as exc:
        print("Star schema audit unavailable:",type(exc).__name__,str(exc)[:250])
        add_check("Star schema audit","NOT VERIFIED",type(exc).__name__,"Dimensions, fact, keys and orphan checks")

## 6. Single Customer View audit

The view should contain one row per identified customer (unknown member excluded). The report records 5,942 customers, 5,852 merchandise purchasers, 90 customers without merchandise orders, snapshot date 2011-12-09, and merchandise revenue £17,117,089.2970. Recency is based on the dataset snapshot, not the system date.

In [ ]:
if engine is not None:
    try:
        with engine.connect() as conn:
            scv = pd.read_sql_query(text("""
                SELECT COUNT(*) AS identified_customer_rows,
                       COUNT(DISTINCT customer_key) AS distinct_customer_keys,
                       COUNT(*) FILTER (WHERE total_orders > 0) AS merchandise_purchasers,
                       COUNT(*) FILTER (WHERE total_orders = 0) AS customers_without_merchandise_orders,
                       SUM(total_merchandise_revenue) AS merchandise_revenue,
                       MAX(snapshot_date) AS snapshot_date
                FROM analytics.vw_single_customer
            """), conn)
        display(scv)
        r = scv.iloc[0]
        compare("SCV identified customers",int(r.identified_customer_rows),BASELINE["scv_customers"])
        compare("SCV distinct customer keys",int(r.distinct_customer_keys),int(r.identified_customer_rows))
        compare("SCV merchandise purchasers",int(r.merchandise_purchasers),BASELINE["purchasers"])
        compare("SCV customers without orders",int(r.customers_without_merchandise_orders),BASELINE["no_orders"])
        compare("SCV merchandise revenue",float(r.merchandise_revenue),BASELINE["scv_revenue"],tol=0.01)
        compare("SCV snapshot date",str(pd.to_datetime(r.snapshot_date).date()),"2011-12-09")
    except Exception as exc:
        print("SCV audit unavailable:",type(exc).__name__,str(exc)[:250])
        add_check("Single Customer View audit","NOT VERIFIED",type(exc).__name__,"analytics.vw_single_customer")

## 7. Repository and SQL asset audit

File presence is evidence that an asset exists, not proof that its migration executed. Runtime database checks above provide separate execution evidence.

In [ ]:
assets = [
    MIGRATIONS/"001_create_staging_table.sql",
    MIGRATIONS/"002_create_cleaned_transactions.sql",
    MIGRATIONS/"003_create_cleaned_transactions_v2.sql",
    MIGRATIONS/"004_create_star_schema.sql",
    MIGRATIONS/"005_create_single_customer_view.sql",
    ERD,BENCHMARK,REPORT
]
asset_df = pd.DataFrame([{"asset":str(p.relative_to(ROOT)),"exists":p.exists(),
                          "size_bytes":p.stat().st_size if p.exists() else None} for p in assets])
display(asset_df)
for _, a in asset_df.iterrows():
    add_check(f"Asset exists: {a.asset}","PASS" if a.exists else "FAIL",bool(a.exists),True)

## 8. SQL performance audit

Historical `EXPLAIN (ANALYZE, BUFFERS, SUMMARY)` results in the Week 1 report:

| Query | Execution time | Target |
|---|---:|---:|
| Single customer lookup (18102) | 1,045.764 ms | under 2,000 ms |
| Top 10 customers by merchandise revenue | 1,544.473 ms | under 2,000 ms |
| Country-level merchandise revenue | 1,891.806 ms | under 2,000 ms |

These are historical measurements, not fresh timings from this notebook. The country query had 108.194 ms headroom in that run. Review the version-controlled benchmark SQL before rerunning it in a controlled session.

In [ ]:
if BENCHMARK.exists():
    print(BENCHMARK.read_text(encoding="utf-8"))
    add_check("Benchmark SQL readable","PASS",True,True)
else:
    add_check("Benchmark SQL readable","FAIL",False,True)

historical = pd.DataFrame([
    ("Single customer lookup (18102)",1045.764),
    ("Top 10 customers by merchandise revenue",1544.473),
    ("Country-level merchandise revenue",1891.806)
],columns=["query","execution_ms"])
historical["target_ms"] = BASELINE["target_ms"]
historical["reported_result"] = historical.execution_ms.map(lambda x:"PASS" if x < BASELINE["target_ms"] else "FAIL")
display(historical)
for _, row in historical.iterrows():
    add_check(f"Historical performance: {row['query']}",row.reported_result,
              float(row.execution_ms),f"< {BASELINE['target_ms']} ms",
              "Historical report evidence; not re-measured in this run.")

## 9. Week 1 requirement traceability

| Week 1 requirement | Evidence reviewed |
|---|---|
| Foundational Star Schema | Dimension/fact counts, fact grain, keys and relationships |
| Clean raw transaction logs | Source reconciliation, staging lineage, quality signals and classification |
| Single Customer View | View grain, customer counts, snapshot and revenue |
| ER Diagram validation | ERD asset plus database FK/orphan checks |
| SQL performance under 2 seconds | Historical EXPLAIN ANALYZE results; fresh run is separate |



## 10. Exceptions and interpretation notes

The Week 1 report documents these caveats:

- Negative quantity is not automatically a confirmed refund; cancellation and negative-quantity flags were handled separately.
- Candidate line amount is not approved accounting revenue by itself.
- Merchandise and non-merchandise amounts are separated.
- Missing customer IDs remain represented by an unknown customer member; they are not fabricated.
- Single Customer View revenue is scoped to identified customers and may differ from the full fact-table merchandise amount.
- The country aggregation was close to the two-second threshold in the historical benchmark.
- Source file and migration file presence do not prove database execution.
- Streamlit Cloud deployment and remote analytical-data hosting are future implementation tasks; they are not evidence of Week 1 completion.

Add issues discovered during execution below. Do not change source data or business definitions merely to force a pass.


## 11. Consolidated audit verdict

`PASS` means every required Week 1 check executed and passed. `FAIL` means at least one required check failed. `INCOMPLETE` means no confirmed failure exists, but evidence is missing or requires review. `NOT VERIFIED` is never a pass.

The deployment inventory is informational and is not included in the Week 1 verdict.


In [ ]:
audit = pd.DataFrame(checks)
display(audit)
if audit.empty:
    verdict = "INCOMPLETE"
else:
    counts = audit.status.value_counts().to_dict()
    if counts.get("FAIL",0):
        verdict = "FAIL"
    elif counts.get("NOT VERIFIED",0) or counts.get("REVIEW",0):
        verdict = "INCOMPLETE"
    else:
        verdict = "PASS"
print("OVERALL WEEK 1 AUDIT VERDICT:",verdict)
print("Check counts:",audit.status.value_counts().to_dict())

## 12. Week 1 audit conclusion

This notebook audits only the Week 1 data foundation. The verdict above is calculated from the checks that could be executed in the current environment.

- `PASS`: all available required checks passed.
- `INCOMPLETE`: required evidence was unavailable or needs review.
- `FAIL`: at least one required control failed.

No manual sign-off fields or exception entries are required. If a database or source file is unavailable, the notebook records that check as `NOT VERIFIED` rather than stopping execution. A `NOT VERIFIED` result is not a pass.
